# Sign a transaction locally

Required environment variables: `EXCHANGEOS_AGENT_PRIVATE_KEY`, `EXCHANGEOS_ASSET_ID`.

Optional: `EXCHANGEOS_ORDER_PRICE`, `EXCHANGEOS_ORDER_SIZE`.

This notebook constructs and signs a Perps limit-order action locally. It does **not** send a REST request and does not need API credentials. Treat the printed signed envelope as sensitive transaction data.

In [ ]:
import os

from exchangeos_sdk import (
    ExchangeOSClient,
    MarginMode,
    OrderSide,
    OrderTimeInForce,
    PlaceOrder,
    generate_client_order_id,
    limit_order_type,
    order_item,
    perp_market,
)

client_order_id = generate_client_order_id()
order = order_item(
    asset_id=os.environ["EXCHANGEOS_ASSET_ID"],
    side=OrderSide.BUY,
    market_type=perp_market(MarginMode.CROSS),
    client_order_id=client_order_id,
    price=os.getenv("EXCHANGEOS_ORDER_PRICE", "50000"),
    size=os.getenv("EXCHANGEOS_ORDER_SIZE", "0.001"),
    order_type=limit_order_type(OrderTimeInForce.GTC),
)
action = PlaceOrder(orders=[order])

In [ ]:
# The default client signs for the PRODUCTION Environment. No network request is made.
with ExchangeOSClient() as client:
    signed_transaction = client.sign_action(
        action,
        os.environ["EXCHANGEOS_AGENT_PRIVATE_KEY"],
    )

print({
    "clientOrderId": client_order_id,
    "signingHash": signed_transaction.tx_hash,
    "signedTransaction": signed_transaction.to_wire(),
})
# To submit this exact envelope later, pass signed_transaction to client.trade.place_order(...).